# HCR 3.0 probe maker
A standalone notebook adapted from RW Null, MD Ramirez, D Sun, and BD Özpolat's HCR 3.0 Probe Maker v0.3.2. Original source: https://github.com/rwnull/insitu_probe_generator; DOI: https://doi.org/10.5281/zenodo.3871970. Original copyright 2019–2021 The Özpolat Lab; GPL-3.0.

Adapted by: Kate Otter

**Run from top to bottom.** Edit configuration first, run the helper cells once, then run **Design**, inspect the candidate sets, and run **Export**. 
To change the sequence or design filters, rerun configuration and Design. 
To change the selected set, BLAST options, or pair limit, rerun configuration and Export.

Input is the **sense cDNA nucleotide sequence**, not a peptide sequence. Each pair uses two antisense 25-base arms separated by two unprobed target bases. Output oligos retain the original initiator orientation and spacers.

The notebook uses supported APIs and calls BLAST+ directly.

## Environment setup — once, outside the notebook
Use a current environment with Python 3.12 or newer, Biopython, pandas, and Jupyter. The notebook uses supported APIs and calls BLAST+ directly.

**Linux/macOS, or Windows through WSL2 (Bioconda):**
```bash
conda create -n hcr-probes --override-channels -c conda-forge -c bioconda --strict-channel-priority python=3.12 biopython pandas jupyterlab ipykernel blast
conda activate hcr-probes
python -m ipykernel install --user --name hcr-probes --display-name "Python (HCR probes)"
jupyter lab
```
Choose the **Python (HCR probes)** kernel. Python 3.12 is a conservative environment target, not a requirement to downgrade newer Python.

**Native Windows:** Bioconda does not support native Windows. For design without BLAST, create the same environment using only `-c conda-forge` and omit `blast`. For BLAST, install NCBI BLAST+ for Windows and set `BLASTN_EXE` to its `blastn.exe` path, or run the full workflow in WSL2. A WSL notebook needs Linux paths such as `/mnt/g/...`, not `G:\...`.

Sources: https://bioconda.github.io/ ; https://bioconda.github.io/faqs.html ; https://biopython.org/wiki/Download ; https://www.ncbi.nlm.nih.gov/books/NBK52637/

In [ ]:
from pathlib import Path
from datetime import datetime
from io import StringIO
from bisect import bisect_left
import hashlib
import json
import re
import shutil
import subprocess
import sys
import Bio
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from IPython.display import display

print(f"Python {sys.version.split()[0]} | Biopython {Bio.__version__} | pandas {pd.__version__}")


## 1. Configuration — edit here
Supply either `SENSE_SEQUENCE` or `INPUT_FASTA`, not both. FASTA files with multiple records require an exact `FASTA_RECORD_ID`. Embedded spaces and newlines in a pasted sequence are removed; U is converted to T. Ambiguous bases are allowed in the input but excluded from candidate windows.

`SKIP_5PRIME_BASES` counts bases at the beginning of the original sense sequence. Optional 3′ trimming occurs before design; zero means no trimming. `MAX_PROBE_PAIRS=None` keeps all selected pairs. The original “choose a longest set” option is now a separate review step with `CHOOSE_ALTERNATIVE_SETS` and `SELECTED_SET`.

In [ ]:
GENE_NAME = "my_gene"
SENSE_SEQUENCE = ""  # Paste the sense cDNA sequence here.
INPUT_FASTA = None  # Example: Path(r"G:\My Drive\project\target.fasta")
FASTA_RECORD_ID = None

AMPLIFIER = "B1"
SKIP_5PRIME_BASES = 100
TRIM_3PRIME_BASES = 0
MAX_POLY_AT = 5
MAX_POLY_CG = 5
MAX_PROBE_PAIRS = 33  # None = no limit.
CHOOSE_ALTERNATIVE_SETS = False
SELECTED_SET = 0
MAX_ALTERNATIVE_SETS = 100

# B1–B5 use the original defined spacers. B7+ originally used ambiguous WW.
# For B7+ enter an explicit two-base A/T spacer for EACH side, e.g. "AA".
SPACER_UP_OVERRIDE = None
SPACER_DOWN_OVERRIDE = None

RUN_BLAST = False
BLAST_SUBJECT_FASTA = None  # Your local nucleotide transcriptome FASTA.
BLASTN_EXE = "blastn"  # Or full path to blastn.exe.
BLAST_TIMEOUT_SECONDS = 600
SHOW_BLAST_DETAILS = False
DROP_LEGACY_FLAGGED_PAIRS = False
REQUIRE_LEGACY_GOOD_HIT = False
WRITE_BLAST_REPORT = False
# Original thresholds retained as configurable legacy heuristics.
GOOD_BITSCORE = 75.0
GOOD_EVALUE_MAX = 1e-13
FLAG_BITSCORE = 60.0
FLAG_EVALUE_MIN = 1e-12

BASE_OUTPUT_DIR = Path("probe_outputs")  # Change to your preferred output folder.
WRITE_PAIR_DETAILS = True
WRITE_RUN_SETTINGS = False


## 2. Helper functions — run once

In [ ]:
AMPLIFIERS = {'B1': {'up': 'GAGGAGGGCAGCAAACGG', 'dn': 'GAAGAGTCTTCCTTTACG', 'upspc': 'aa', 'dnspc': 'ta'}, 'B2': {'up': 'CCTCGTAAATCCTCATCA', 'dn': 'ATCATCCAGTAAACCGCC', 'upspc': 'aa', 'dnspc': 'aa'}, 'B3': {'up': 'GTCCCTGCCTCTATATCT', 'dn': 'CCACTCAACTTTAACCCG', 'upspc': 'tt', 'dnspc': 'tt'}, 'B4': {'up': 'CCTCAACCTACCTCCAAC', 'dn': 'TCTCACCATATTCGCTTC', 'upspc': 'aa', 'dnspc': 'at'}, 'B5': {'up': 'CTCACTCCCAATCTCTAT', 'dn': 'CTACCCTACAAATCCAAT', 'upspc': 'aa', 'dnspc': 'aa'}, 'B7': {'up': 'CTTCAACCTCCACCTACC', 'dn': 'TCCAATCCCTACCCTCAC', 'upspc': 'ww', 'dnspc': 'ww'}, 'B9': {'up': 'CACGTATCTACTCCACTC', 'dn': 'TCAGCACACTCCCAACCC', 'upspc': 'ww', 'dnspc': 'ww'}, 'B10': {'up': 'CCTCAAGATACTCCTCTA', 'dn': 'CCTACTCGACTACCCTAG', 'upspc': 'ww', 'dnspc': 'ww'}, 'B11': {'up': 'CGCTTAGATATCACTCCT', 'dn': 'ACGTCGACCACACTCATC', 'upspc': 'ww', 'dnspc': 'ww'}, 'B13': {'up': 'AGGTAACGCCTTCCTGCT', 'dn': 'TTATGCTCAACATACAAC', 'upspc': 'ww', 'dnspc': 'ww'}, 'B14': {'up': 'AATGTCAATAGCGAGCGA', 'dn': 'CCCTATATTTCTGCACAG', 'upspc': 'ww', 'dnspc': 'ww'}, 'B15': {'up': 'CAGATTAACACACCACAA', 'dn': 'GGTATCTCGAACACTCTC', 'upspc': 'ww', 'dnspc': 'ww'}, 'B17': {'up': 'CGATTGTTTGTTGTGGAC', 'dn': 'GCATGCTAATCGGATGAG', 'upspc': 'ww', 'dnspc': 'ww'}}

In [ ]:
def load_sense_sequence(sequence, fasta_path=None, record_id=None, trim_3prime=0):
    pasted = re.sub(r"\s+", "", str(sequence or ""))
    if bool(pasted) == bool(fasta_path):
        raise ValueError("Supply either SENSE_SEQUENCE or INPUT_FASTA, not both or neither.")
    if fasta_path:
        path = Path(fasta_path)
        if not path.is_file():
            raise FileNotFoundError(f"Input FASTA not found: {path}")
        records = list(SeqIO.parse(path, "fasta"))
        if record_id is not None:
            records = [r for r in records if r.id == record_id]
        if len(records) != 1:
            raise ValueError(f"Found {len(records)} records. Set FASTA_RECORD_ID to one exact, unique ID.")
        pasted = str(records[0].seq)
    sequence = pasted.upper().replace("U", "T")
    invalid = set(sequence) - set("ACGTRYSWKMBDHVN")
    if invalid:
        raise ValueError(f"Invalid nucleotide characters: {sorted(invalid)}. Do not paste FASTA headers here.")
    if isinstance(trim_3prime, bool) or not isinstance(trim_3prime, int) or not 0 <= trim_3prime < len(sequence):
        raise ValueError("TRIM_3PRIME_BASES must be an integer from zero to less than sequence length.")
    return sequence[:-trim_3prime] if trim_3prime else sequence


def validate_design_settings(sequence, pause, poly_at, poly_cg):
    for name, value in [("SKIP_5PRIME_BASES", pause), ("MAX_POLY_AT", poly_at), ("MAX_POLY_CG", poly_cg)]:
        if isinstance(value, bool) or not isinstance(value, int) or value < (0 if name == "SKIP_5PRIME_BASES" else 1):
            raise ValueError(f"{name} must be a valid nonnegative/positive integer.")
    if len(sequence) - pause < 52:
        raise ValueError("Fewer than 52 target bases remain after the 5′ skip and 3′ trim.")


def resolve_amplifier(amplifier, spacer_up=None, spacer_down=None):
    key = str(amplifier).upper().strip()
    if key not in AMPLIFIERS:
        raise ValueError(f"Unknown amplifier {key}; choose {', '.join(AMPLIFIERS)}.")
    entry = AMPLIFIERS[key].copy()
    for field, override in [("upspc", spacer_up), ("dnspc", spacer_down)]:
        spacer = str(override if override is not None else entry[field]).upper()
        if len(spacer) != 2 or set(spacer) - set("AT"):
            raise ValueError(f"{key} {field} is {spacer!r}. Set its spacer override to two explicit A/T bases.")
        entry[field] = spacer.lower()
    return entry


def candidate_windows(sequence, pause, poly_at, poly_cg):
    validate_design_settings(sequence, pause, poly_at, poly_cg)
    antisense = str(Seq(sequence).reverse_complement())
    forbidden = [base * (limit + 1) for base, limit in [("A", poly_at), ("T", poly_at), ("C", poly_cg), ("G", poly_cg)]]
    starts = []
    # Inclusive last window fixes the old code's missing final/exactly-52-base window.
    for start in range(len(sequence) - pause - 52 + 1):
        window = antisense[start:start + 52]
        if set(window) <= set("ACGT") and not any(run in window for run in forbidden):
            starts.append(start)
    return antisense, starts


def select_candidate_sets(starts, alternatives=False, max_sets=100):
    if not starts:
        return []
    if isinstance(max_sets, bool) or not isinstance(max_sets, int) or max_sets < 1:
        raise ValueError("MAX_ALTERNATIVE_SETS must be a positive integer.")
    def greedy(first):
        chosen = [starts[first]]
        index = first
        while True:
            # Preserve original condition: next start > previous end + 2.
            index = bisect_left(starts, chosen[-1] + 55, lo=index + 1)
            if index >= len(starts):
                return chosen
            chosen.append(starts[index])
    first = greedy(0)
    if not alternatives:
        return [first]
    longest = len(first)  # Fixed-size intervals: earliest-finish greedy gives maximum count.
    sets = [first]
    for index in range(1, len(starts)):
        if len(starts) - index < longest:
            break
        chosen = greedy(index)
        if len(chosen) == longest:
            sets.append(chosen)
            if len(sets) >= max_sets:
                print(f"Showing the first {max_sets} equally large sets; later alternatives were not listed.")
                break
    return sets


PAIR_COLUMNS = ["pair_id", "antisense_start_0", "target_start_1", "target_end_1",
                "up_target_start_1", "up_target_end_1", "down_target_start_1", "down_target_end_1",
                "up_binding", "down_binding", "blast_query"]


def build_pairs(sequence, antisense, starts):
    n = len(sequence)
    rows = []
    for index, start in enumerate(starts, 1):
        down = antisense[start:start + 25]
        up = antisense[start + 27:start + 52]
        rows.append({
            "pair_id": f"P{index:04d}", "antisense_start_0": start,
            "target_start_1": n - start - 51, "target_end_1": n - start,
            "up_target_start_1": n - start - 51, "up_target_end_1": n - start - 27,
            "down_target_start_1": n - start - 24, "down_target_end_1": n - start,
            "up_binding": up, "down_binding": down,
            "blast_query": down + "NN" + up,
        })
    return pd.DataFrame(rows, columns=PAIR_COLUMNS)


def limit_pairs(pairs, max_pairs):
    if max_pairs is None:
        return pairs.copy()
    if isinstance(max_pairs, bool) or not isinstance(max_pairs, int) or max_pairs < 1:
        raise ValueError("MAX_PROBE_PAIRS must be None or a positive integer.")
    if len(pairs) <= max_pairs:
        return pairs.copy()
    # Spread the retained pairs across the designed region.
    indices = [round(i * (len(pairs) - 1) / (max_pairs - 1)) for i in range(max_pairs)] if max_pairs > 1 else [0]
    return pairs.iloc[indices].copy().reset_index(drop=True)


## 3. Optional local BLAST helpers
BLAST is off by default. When enabled, it searches the selected 52-base pair queries (two arms joined by NN) against your nucleotide FASTA, using `blastn-short` as in the original script. An empty successful BLAST output means zero hits, not a software failure.

In [ ]:
BLAST_COLUMNS = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
                 "qstart", "qend", "sstart", "send", "evalue", "bitscore"]


def parse_blast_table(text):
    if not text.strip():
        return pd.DataFrame(columns=BLAST_COLUMNS)
    table = pd.read_csv(StringIO(text), sep="\t", header=None,
                        names=BLAST_COLUMNS, dtype={"qseqid": str, "sseqid": str})
    if table.isna().any().any():
        raise ValueError("Malformed BLAST table: expected 12 fields per row.")
    for column in BLAST_COLUMNS[2:]:
        table[column] = pd.to_numeric(table[column], errors="raise")
    return table


def run_local_blast(pairs, subject_fasta, executable="blastn", timeout=600):
    import tempfile
    if subject_fasta is None or not Path(subject_fasta).is_file():
        raise FileNotFoundError("Set BLAST_SUBJECT_FASTA to an existing nucleotide FASTA.")
    exe = shutil.which(str(executable))
    if exe is None:
        raise FileNotFoundError(f"BLAST+ executable not found: {executable}. Install BLAST+ or set BLASTN_EXE.")
    if pairs.empty:
        return pd.DataFrame(columns=BLAST_COLUMNS)
    with tempfile.TemporaryDirectory(prefix="hcr_blast_") as directory:
        query = Path(directory) / "pairs.fasta"
        SeqIO.write([SeqRecord(Seq(row.blast_query), id=row.pair_id, description="")
                     for row in pairs.itertuples()], query, "fasta")
        command = [exe, "-query", str(query), "-subject", str(Path(subject_fasta).resolve()),
                   "-task", "blastn-short", "-outfmt", "6 " + " ".join(BLAST_COLUMNS)]
        try:
            result = subprocess.run(command, capture_output=True, text=True, check=True, timeout=timeout)
        except subprocess.CalledProcessError as error:
            raise RuntimeError(f"BLAST failed (exit {error.returncode}): {error.stderr.strip()}") from error
        if result.stderr.strip():
            print("BLAST message:", result.stderr.strip())
    return parse_blast_table(result.stdout)


def screen_legacy_blast(pairs, hits, drop_flagged=False, require_good=False,
                        good_bitscore=75.0, good_evalue=1e-13,
                        flag_bitscore=60.0, flag_evalue=1e-12):
    if not set(hits["qseqid"]).issubset(set(pairs["pair_id"])):
        raise ValueError("BLAST results contain unexpected pair IDs.")
    good_mask = (hits["bitscore"] >= good_bitscore) & (hits["evalue"] <= good_evalue)
    flag_mask = (~good_mask) & (hits["bitscore"] >= flag_bitscore) & (hits["evalue"] > flag_evalue)
    good = set(hits.loc[good_mask, "qseqid"])
    flagged = set(hits.loc[flag_mask, "qseqid"])
    annotated = pairs.copy()
    annotated["legacy_good_hit"] = annotated["pair_id"].isin(good)
    annotated["legacy_flagged_hit"] = annotated["pair_id"].isin(flagged)
    keep = pd.Series(True, index=annotated.index)
    if drop_flagged:
        keep &= ~annotated["legacy_flagged_hit"]
    if require_good:
        keep &= annotated["legacy_good_hit"]
    print(f"BLAST: {len(hits)} alignments; {len(good)} pairs with legacy good hits; {len(flagged)} flagged pairs.")
    return annotated.loc[keep].copy().reset_index(drop=True)


def make_order_table(pairs, name, amplifier, settings, pause):
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_")
    if not safe_name:
        raise ValueError("GENE_NAME must contain at least one letter or digit.")
    pool = f"{amplifier.upper()}_{safe_name}_{len(pairs)}_Dla{pause}"
    rows = []
    for row in pairs.itertuples():
        rows.append({"Pool name": pool, "Sequence": settings["up"] + settings["upspc"] + row.up_binding})
        rows.append({"Pool name": pool, "Sequence": row.down_binding + settings["dnspc"] + settings["dn"]})
    return pd.DataFrame(rows, columns=["Pool name", "Sequence"])


## 4. Design — run after configuration and helpers

In [ ]:
sense_sequence = load_sense_sequence(SENSE_SEQUENCE, INPUT_FASTA, FASTA_RECORD_ID, TRIM_3PRIME_BASES)
amplifier_settings = resolve_amplifier(AMPLIFIER, SPACER_UP_OVERRIDE, SPACER_DOWN_OVERRIDE)
antisense_sequence, valid_starts = candidate_windows(sense_sequence, SKIP_5PRIME_BASES, MAX_POLY_AT, MAX_POLY_CG)
probe_sets = select_candidate_sets(valid_starts, CHOOSE_ALTERNATIVE_SETS, MAX_ALTERNATIVE_SETS)
# Remember design inputs, so changed settings cannot silently reuse an old design.
design_signature = (sense_sequence, SKIP_5PRIME_BASES, MAX_POLY_AT, MAX_POLY_CG,
                    CHOOSE_ALTERNATIVE_SETS, MAX_ALTERNATIVE_SETS)
print(f"Target: {len(sense_sequence)} bases; valid windows: {len(valid_starts)}; available sets: {len(probe_sets)}")
set_summary = pd.DataFrame([
    {"set_index": i, "pairs": len(starts),
     "first_target_base": len(sense_sequence) - starts[-1] - 51,
     "last_target_base": len(sense_sequence) - starts[0]}
    for i, starts in enumerate(probe_sets)
])
display(set_summary)
if not probe_sets:
    print("No candidates passed. Review sequence, skipped bases, and homopolymer limits; nothing will be exported.")
else:
    display(build_pairs(sense_sequence, antisense_sequence, probe_sets[0]).drop(columns=["blast_query", "antisense_start_0"]).head(10))


## 5. export — run after reviewing Design
Writes one **ordering CSV** with `Pool name` and `Sequence`, plus one optional **pair-details CSV**. BLAST report and run-settings JSON are optional. Each execution creates a new run folder, so earlier orders are not overwritten.

In [ ]:
current_sequence = load_sense_sequence(SENSE_SEQUENCE, INPUT_FASTA, FASTA_RECORD_ID, TRIM_3PRIME_BASES)
current_signature = (current_sequence, SKIP_5PRIME_BASES, MAX_POLY_AT, MAX_POLY_CG,
                     CHOOSE_ALTERNATIVE_SETS, MAX_ALTERNATIVE_SETS)
if "design_signature" not in globals() or current_signature != design_signature:
    raise RuntimeError("Run Design again after changing the sequence or design filters.")
if not probe_sets:
    raise ValueError("No valid probe pairs. Nothing exported.")
if isinstance(SELECTED_SET, bool) or not isinstance(SELECTED_SET, int) or not 0 <= SELECTED_SET < len(probe_sets):
    raise ValueError(f"SELECTED_SET must be 0 to {len(probe_sets) - 1}.")
amplifier_settings = resolve_amplifier(AMPLIFIER, SPACER_UP_OVERRIDE, SPACER_DOWN_OVERRIDE)
selected_pairs = build_pairs(sense_sequence, antisense_sequence, probe_sets[SELECTED_SET])
blast_hits = pd.DataFrame(columns=BLAST_COLUMNS)
if RUN_BLAST:
    blast_hits = run_local_blast(selected_pairs, BLAST_SUBJECT_FASTA, BLASTN_EXE, BLAST_TIMEOUT_SECONDS)
    selected_pairs = screen_legacy_blast(
        selected_pairs, blast_hits, DROP_LEGACY_FLAGGED_PAIRS, REQUIRE_LEGACY_GOOD_HIT,
        GOOD_BITSCORE, GOOD_EVALUE_MAX, FLAG_BITSCORE, FLAG_EVALUE_MIN,
    )
    if SHOW_BLAST_DETAILS:
        display(blast_hits)
selected_pairs = limit_pairs(selected_pairs, MAX_PROBE_PAIRS)
if selected_pairs.empty:
    raise ValueError("No pairs remain after BLAST filtering. Nothing exported.")
order_table = make_order_table(selected_pairs, GENE_NAME, AMPLIFIER, amplifier_settings, SKIP_5PRIME_BASES)
safe_gene = re.sub(r"[^A-Za-z0-9_.-]+", "_", GENE_NAME).strip("_")
run_dir = Path(BASE_OUTPUT_DIR) / f"{safe_gene}_{datetime.now():%Y%m%d_%H%M%S_%f}"
run_dir.mkdir(parents=True, exist_ok=False)
order_path = run_dir / f"{safe_gene}_{AMPLIFIER.upper()}_order.csv"
order_table.to_csv(order_path, index=False)
print(f"Order: {order_path.resolve()} ({len(selected_pairs)} pairs; {len(order_table)} oligos)")
if WRITE_PAIR_DETAILS:
    pair_path = run_dir / f"{safe_gene}_{AMPLIFIER.upper()}_pairs.csv"
    selected_pairs.drop(columns=["blast_query", "antisense_start_0"]).to_csv(pair_path, index=False)
    print(f"Pair details: {pair_path.resolve()}")
if RUN_BLAST and WRITE_BLAST_REPORT:
    blast_hits.to_csv(run_dir / "blast_hits.csv", index=False)
if WRITE_RUN_SETTINGS:
    settings = {
        "gene": GENE_NAME, "amplifier": AMPLIFIER, "spacer_up": amplifier_settings["upspc"],
        "spacer_down": amplifier_settings["dnspc"], "input_fasta": str(INPUT_FASTA) if INPUT_FASTA else None,
        "record_id": FASTA_RECORD_ID, "sequence_sha256": hashlib.sha256(sense_sequence.encode()).hexdigest(),
        "sequence_length": len(sense_sequence), "skip_5prime": SKIP_5PRIME_BASES,
        "trim_3prime": TRIM_3PRIME_BASES, "poly_at": MAX_POLY_AT, "poly_cg": MAX_POLY_CG,
        "selected_set": SELECTED_SET, "max_pairs": MAX_PROBE_PAIRS, "pairs_exported": len(selected_pairs),
        "blast_enabled": RUN_BLAST, "blast_subject": str(BLAST_SUBJECT_FASTA) if BLAST_SUBJECT_FASTA else None,
        "drop_flagged": DROP_LEGACY_FLAGGED_PAIRS, "require_good": REQUIRE_LEGACY_GOOD_HIT,
        "good_bitscore": GOOD_BITSCORE, "good_evalue": GOOD_EVALUE_MAX,
        "flag_bitscore": FLAG_BITSCORE, "flag_evalue": FLAG_EVALUE_MIN,
        "python": sys.version, "biopython": Bio.__version__, "pandas": pd.__version__,
    }
    (run_dir / "run_settings.json").write_text(json.dumps(settings, indent=2))
display(order_table)
